# 의학 분야 ChromaDB/FAISS 인덱싱 + QA Chain 실습

## 0. 환경 설정 및 패키지 설치

In [1]:
# 1) 나눔고딕 폰트 설치
!apt-get -qq update
!apt-get -qq install -y fonts-nanum

# 2) matplotlib 폰트 캐시 갱신
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')

import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 부호 깨짐 방지

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [3]:
!pip install -q sentence-transformers chromadb faiss-cpu langchain langchain-community langchain-huggingface

In [ ]:
#원래 코드가 reports를  dictionary의 list로 처리

- 실습용 데이터 로드

In [ ]:
from sentence_transformers import SentenceTransformer
from datasets import load_dataset

ds = load_dataset("allganize/RAG-Evaluation-Dataset-KO")
df = ds["test"].to_pandas()

print(df.shape)          # (300, ~25+)
print(df.columns.tolist())
print(df["domain"].unique())  # 금융 / 공공 / 의료 / 법률 / 커머스
df[["domain", "question", "target_answer"]].head(5)

README.md:   0%|          | 0.00/11.4k [00:00<?, ?B/s]

rag_evaluation_result.csv:   0%|          | 0.00/5.69M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/300 [00:00<?, ? examples/s]

(300, 52)
['domain', 'question', 'target_answer', 'target_file_name', 'target_page_no', 'context_type', 'alli_gpt-4-turbo_answer', 'alli_gpt-4-turbo_ox', 'alli_gpt-4_answer', 'alli_gpt-4_ox', 'alli_claude3-opus_answer', 'alli_claude3-opus_ox', 'alli_Llama-3-Ko-8B-Finance-Evol_answer', 'alli_Llama-3-Ko-8B-Finance-Evol_ox', 'alli_Llama-3-Alpha-Ko-8B-Instruct_answer', 'alli_Llama-3-Alpha-Ko-8B-Instruct_ox', 'alli_KONI-Llama3-8B-Instruct-20240729_answer', 'alli_KONI-Llama3-8B-Instruct-20240729_ox', 'alli_claude-3.5-sonnet_answer', 'alli_claude-3.5-sonnet_ox', 'alli_gpt-4o_answer', 'alli_gpt-4o_ox', 'alli_gpt-4o-mini_answer', 'alli_gpt-4o-mini_ox', 'alli_alpha-ko-202411-32B_answer', 'alli_alpha-ko-202411-32B_ox', 'langchain_gpt-4-turbo_answer', 'langchain_gpt-4-turbo_ox', 'langchain_gpt-3.5-turbo_answer', 'langchain_gpt-3.5-turbo_ox', 'openai_assistant_gpt-4-turbo_answer', 'openai_assistant_gpt-4-turbo_ox', 'openai_assistant_gpt-4_answer', 'openai_assistant_gpt-4_ox', 'cohere_command-r_answ

,domain,question,target_answer
0,finance,"시중은행, 지방은행, 인터넷은행의 인가 요건 및 절차에 차이가 있는데 그 차이점은 ...","시중은행, 지방은행, 인터넷은행 모두 은행업을 영위하기 위해서는 '은행법' 제8조에..."
1,finance,"은행업을 신청하고자 할 때, 은행법상 소유규제에 부합하는 대주주 요건을 충족하려면 ...",은행업을 신청하려면 대주주 요건을 충족해야 합니다. 대주주 요건으로는 부실금융기관 ...
2,finance,본인가를 받으려는 지방은행이 시중은행 전환시 예비인가를 받을 필요가 있는지 설명하시...,"본인가를 받으려는 지방은행이 시중은행 전환을 신청하는 경우, 예비인가를 받을 필요는..."
3,finance,"은행법에 의거 예비인가를 신청할 수 있는지와, 그 경우 금융위원회가 검토했어야 하는...",은행법에 의하면 예비인가를 신청할 수 있습니다. 제8조에 따른 인가를 받으려는 자는...
4,finance,2019년 YTD 기준으로 브라질의 주식 시장 수익률과 베트남의 주식 시장 수익률 ...,Refinitiv에서 제공한 자료에 따르면 2019년 YTD 브라질의 주식 시장 수...


In [ ]:
df["domain"].value_counts()
df["ids"] = ["id"+ str(i+1) for i in df.index]
# question → RAG 검색 쿼리로 활용
# target_answer → 정답 비교 (평가 지표 계산)


In [ ]:
# 도메인별 필터링 — 의료 분야로 필터링
df_medical = df[df["domain"] == "medical"]  #df_finance = df[df["domain"] == "finance"]
reports = df_medical.to_dict(orient="records")